[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/24_rope.ipynb)

# 🔴 Hard: Rotary Position Embedding (RoPE)

Implement **RoPE** — the position encoding used in LLaMA, GPT-NeoX, and most modern LLMs.

### Signature
```python
def apply_rope(q: Tensor, k: Tensor) -> tuple[Tensor, Tensor]:
    # q, k: (B, S, D) where D is even
    # Returns rotated (q, k) with same shape
```

### Key Idea
Split each vector into consecutive pairs. Rotate each pair by `θ = pos / 10000^(2i/D)`:
```
[x_0, x_1] → [x_0*cosθ - x_1*sinθ, x_0*sinθ + x_1*cosθ]
```
This makes `dot(q_rot[i], k_rot[j])` depend only on `i - j` (relative position).

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.0 MB/s eta 0:00:00


In [2]:
import torch
import math

In [23]:
# ✏️ YOUR IMPLEMENTATION HERE

def apply_rope(q, k):
    S = q.shape[1]
    D = q.shape[2]
    # 1. Compute position angles
    row = torch.arange(S).view(S, 1)  # (m, 1)
    col = torch.arange(D//2).view(1, D//2)  # (1, n)
    theta = row / 1000 ** (2 * col / D)
    assert theta.shape == (S, D//2)
    theta = torch.unsqueeze(theta, 0)
    assert theta.shape == (1, S, D//2)
    # 2. Split into even/odd pairs
    (q1, q2) = torch.split(q, D//2, dim=2)
    (k1, k2) = torch.split(k, D//2, dim=2)

    # 3. Apply rotation
    cost = torch.cos(theta)
    sint = torch.sin(theta)

    qq1 = q1 * cost - q2 * sint
    qq2 = q1 * sint + q2 * cost

    kk1 = k1 * cost - k2 * sint
    kk2 = k1 * sint + k2 * cost

    Q = torch.concat([qq1, qq2], dim=-1)
    K = torch.concat([kk1, kk2], dim=-1)

    return (Q, K)




In [24]:
# 🧪 Debug
q = torch.randn(1, 8, 16)
k = torch.randn(1, 8, 16)
qr, kr = apply_rope(q, k)
print('Shape preserved:', qr.shape == q.shape)
print('Norm preserved:', torch.allclose(q.norm(dim=-1), qr.norm(dim=-1), atol=1e-4))

Shape preserved: True
Norm preserved: True


In [25]:
# ✅ SUBMIT
from torch_judge import check
check('rope')


🧪 Testing: Rotary Position Embedding (RoPE) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shapes (0.8ms)
  ✅ [2/4] Preserves norm (5.1ms)
  ✅ [3/4] Relative position property (7.5ms)
  ✅ [4/4] Gradient flow (12.3ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (25.7ms total)
  Progress saved. Run status() to see your dashboard.

